In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
def trans_conv(X, K):
    """转置卷积step=1时的实现方法"""
    h, w = K.shape # 获得kernel的高宽
    Y = torch.zeros((X.shape[0] + h - 1, X.shape[1] + w - 1)) # 创建一个(X_h + k_h - 1, X_w + k_w - 1)的矩阵，这就是最终的矩阵大小
    for i in range(X.shape[0]): # 在X轴上滑动
        for j in range(X.shape[1]): # 在Y轴上滑动
            Y[i: i + h, j: j + w] += X[i, j] * K # X(i, j)的值（标量）跟kernel相乘，放到Y(i, j)上
    return Y

In [3]:
X = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
trans_conv(X, K)

tensor([[ 0.,  0.,  1.],
        [ 0.,  4.,  6.],
        [ 4., 12.,  9.]])

In [4]:
X, K = X.reshape(1, 1, 2, 2), K.reshape(1, 1, 2, 2)
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, bias=False) # 转置卷积的简介实现nn.ConvTranspose2d(输入通道，输出通道，卷积核大小，是否有偏移)
tconv.weight.data = K
tconv(X)

tensor([[[[ 0.,  0.,  1.],
          [ 0.,  4.,  6.],
          [ 4., 12.,  9.]]]], grad_fn=<ConvolutionBackward0>)

In [5]:
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, padding=1, bias=False)
tconv.weight.data = K
tconv(X)

tensor([[[[4.]]]], grad_fn=<ConvolutionBackward0>)

In [6]:
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2, bias=False)
tconv.weight.data = K
tconv(X)

tensor([[[[0., 0., 0., 1.],
          [0., 0., 2., 3.],
          [0., 2., 0., 3.],
          [4., 6., 6., 9.]]]], grad_fn=<ConvolutionBackward0>)

In [7]:
X = torch.rand(size=(1, 10, 16, 16))
conv = nn.Conv2d(10, 20, kernel_size=5, padding=2, stride=3)
tconv = nn.ConvTranspose2d(20, 10, kernel_size=5, padding=2, stride=3)
tconv(conv(X)).shape == X.shape # 同样超参数的卷积和转置卷积，shape是可以互相转换的

True

转置卷积的padding可以理解为，是在最终的输出结果上padding，然后在通过卷积之后获得的当下的值。所以padding越大，最终输出的尺寸就越小。